# C9-dimensionality-reduction — Practice p22

**Type:** constrained coding · **Difficulty:** core · **Concepts:** numpy-pca-class-from-scratch, pca-black-box-insufficiency

Implement `NumpyPCA(n_components)` using NumPy only.

`fit(X)` must accept a finite numeric matrix `X` of shape `(n, d)` with `n >= 2`, `d >= 1`, and integer `1 <= n_components <= d`.
Reject every invalid case with `ValueError` before decomposition.
Compute `mean_`, center with the training mean, form exactly
`covariance = Xc.T @ Xc / (n - 1)`.
If this intermediate covariance is non-finite, raise `ValueError` before decomposition.
Otherwise make exactly one `np.linalg.eigh(covariance)` call; if its returned eigenvalues or eigenvectors are non-finite, raise `ValueError` before committing learned state.
Sort finite eigenpairs together into descending variance order and return `self`.
Fit is atomic: any failed first fit leaves the model unfitted, and any failed refit preserves all four arrays from the previous successful fit.

After fit, store finite float arrays:

- `mean_`, shape `(d,)`;
- `components_`, shape `(k, d)`, orthonormal rows;
- `explained_variance_`, shape `(k,)`;
- `explained_variance_ratio_`, shape `(k,)`, using the sum of the **full** covariance spectrum as denominator.

For zero total variance, define the ratio vector as zeros.
`transform(X_new)` must reject use before fit, non-finite/non-numeric/non-matrix input, empty rows, or a wrong feature count.
It returns `(X_new - mean_) @ components_.T` and must not recompute a mean or fit state.
Do not mutate inputs.

**Zero points:** sklearn PCA, scipy PCA, or any helper that invokes them.
The checker exposes its fixed matrices, instruments the `eigh` call, checks refit state, and certifies values independently through covariance and centered SVD.
Use `ATOL = 1e-10`, `RTOL = 0.0`.


In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0


class NumpyPCA:
    def __init__(self, n_components):
        # YOUR CODE HERE
        ...

    def fit(self, X):
        # YOUR CODE HERE
        ...

    def transform(self, X):
        # YOUR CODE HERE
        ...

## Immutable contract check — do not edit

The check uses public fixtures and explicit tolerances.
It compares signs through one-dimensional projectors, verifies the full-spectrum ratio denominator, and checks that transform reuses fitted state.


In [ ]:
import dis
import inspect
import types

_ORIGINAL_EIGH_P22 = np.linalg.eigh


def _audit_numpy_pca_p22():
    pending = []
    for value in vars(NumpyPCA).values():
        if isinstance(value, (staticmethod, classmethod)):
            value = value.__func__
        if isinstance(value, types.FunctionType):
            pending.append(value)
    seen = set()
    while pending:
        function = pending.pop()
        if id(function) in seen:
            continue
        seen.add(id(function))
        codes = [function.__code__]
        while codes:
            code = codes.pop()
            codes.extend(
                item for item in code.co_consts
                if isinstance(item, types.CodeType)
            )
            names = {name.lower() for name in code.co_names}
            assert not ({"sklearn", "scipy"} & names)
            for instruction in dis.get_instructions(code):
                if instruction.opname in {"IMPORT_NAME", "IMPORT_FROM"}:
                    assert not str(instruction.argval).lower().startswith(("sklearn", "scipy"))
            for name in code.co_names:
                value = function.__globals__.get(name)
                if isinstance(value, types.FunctionType):
                    pending.append(value)
                module_name = str(getattr(value, "__module__", "")).lower()
                object_name = str(getattr(value, "__name__", "")).lower()
                assert not module_name.startswith(("sklearn", "scipy"))
                assert not object_name.startswith(("sklearn", "scipy"))
    try:
        source = inspect.getsource(NumpyPCA).lower()
    except (OSError, TypeError):
        source = ""
    assert "sklearn" not in source and "scipy" not in source


_audit_numpy_pca_p22()

_X_train_p22 = np.array([
    [4.0, 1.0, 2.0],
    [2.0, 3.0, 0.0],
    [0.0, 1.0, 2.0],
    [2.0, -1.0, 4.0],
    [5.0, 2.0, 1.5],
])
_X_new_p22 = np.array([[8.0, -2.0, 1.0], [1.0, 5.0, -3.0]])


def _fit_with_eigh_trace_p22(model, X):
    calls = []
    def traced(matrix):
        calls.append(np.array(matrix, copy=True))
        return _ORIGINAL_EIGH_P22(matrix)
    np.linalg.eigh = traced
    try:
        result = model.fit(X)
    finally:
        np.linalg.eigh = _ORIGINAL_EIGH_P22
    return result, calls


_before_train_p22 = _X_train_p22.copy()
_model_p22 = NumpyPCA(2)
_returned_p22, _eigh_calls_p22 = _fit_with_eigh_trace_p22(_model_p22, _X_train_p22)
assert _returned_p22 is _model_p22
assert np.array_equal(_X_train_p22, _before_train_p22)
assert len(_eigh_calls_p22) == 1
_mean_ref_p22 = _X_train_p22.mean(axis=0)
_Xc_ref_p22 = _X_train_p22 - _mean_ref_p22
_C_ref_p22 = _Xc_ref_p22.T @ _Xc_ref_p22 / (_X_train_p22.shape[0] - 1)
assert np.allclose(_eigh_calls_p22[0], _C_ref_p22, atol=ATOL, rtol=RTOL)
_evals_ref_p22, _evecs_ref_p22 = _ORIGINAL_EIGH_P22(_C_ref_p22)
_order_ref_p22 = np.argsort(_evals_ref_p22)[::-1]
_evals_ref_p22 = np.maximum(_evals_ref_p22[_order_ref_p22], 0.0)
_evecs_ref_p22 = _evecs_ref_p22[:, _order_ref_p22]
_total_ref_p22 = _evals_ref_p22.sum()

for _name_p22, _shape_p22 in (
    ("mean_", (3,)),
    ("components_", (2, 3)),
    ("explained_variance_", (2,)),
    ("explained_variance_ratio_", (2,)),
):
    _value_p22 = getattr(_model_p22, _name_p22)
    assert isinstance(_value_p22, np.ndarray) and _value_p22.shape == _shape_p22
    assert np.issubdtype(_value_p22.dtype, np.floating)
    assert np.isfinite(_value_p22).all()
assert np.allclose(_model_p22.mean_, _mean_ref_p22, atol=ATOL, rtol=RTOL)
assert np.allclose(
    _model_p22.components_ @ _model_p22.components_.T, np.eye(2),
    atol=ATOL, rtol=RTOL,
)
assert np.allclose(_model_p22.explained_variance_, _evals_ref_p22[:2], atol=ATOL, rtol=RTOL)
assert np.allclose(
    _model_p22.explained_variance_ratio_, _evals_ref_p22[:2] / _total_ref_p22,
    atol=ATOL, rtol=RTOL,
)
for _j_p22 in range(2):
    _P_student_p22 = np.outer(_model_p22.components_[_j_p22], _model_p22.components_[_j_p22])
    _P_ref_p22 = np.outer(_evecs_ref_p22[:, _j_p22], _evecs_ref_p22[:, _j_p22])
    assert np.allclose(_P_student_p22, _P_ref_p22, atol=ATOL, rtol=RTOL)

_before_new_p22 = _X_new_p22.copy()
_scores_p22 = _model_p22.transform(_X_new_p22)
assert np.array_equal(_X_new_p22, _before_new_p22)
assert isinstance(_scores_p22, np.ndarray) and _scores_p22.shape == (2, 2)
assert np.issubdtype(_scores_p22.dtype, np.floating) and np.isfinite(_scores_p22).all()
assert np.allclose(
    _scores_p22, (_X_new_p22 - _mean_ref_p22) @ _model_p22.components_.T,
    atol=ATOL, rtol=RTOL,
)

# Refit changes both the origin and covariance. Controlled eigenpairs make the
# returned values, rather than a dummy eigh call followed by another decomposition,
# the only route to the required learned state.
_X_refit_p22 = np.array([
    [31.0, -8.0, 2.0], [27.0, -3.0, 9.0], [35.0, -5.0, -1.0],
    [29.0, -11.0, 6.0], [40.0, -1.0, 4.0], [25.0, -7.0, 12.0],
])
_mean_refit_p22 = _X_refit_p22.mean(axis=0)
_Xc_refit_p22 = _X_refit_p22 - _mean_refit_p22
_C_refit_p22 = _Xc_refit_p22.T @ _Xc_refit_p22 / (_X_refit_p22.shape[0] - 1)
assert not np.allclose(_C_refit_p22, _C_ref_p22, atol=ATOL, rtol=RTOL)
_sentinel_values_p22 = np.array([4.25, 12.5, 7.75])
_sentinel_vectors_p22 = np.array([
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
    [1.0, 0.0, 0.0],
])
_controlled_calls_p22 = []
def _controlled_eigh_p22(matrix):
    _controlled_calls_p22.append(np.array(matrix, copy=True))
    assert np.allclose(matrix, _C_refit_p22, atol=ATOL, rtol=RTOL)
    return _sentinel_values_p22.copy(), _sentinel_vectors_p22.copy()
_before_refit_p22 = _X_refit_p22.copy()
np.linalg.eigh = _controlled_eigh_p22
try:
    _returned_refit_p22 = _model_p22.fit(_X_refit_p22)
finally:
    np.linalg.eigh = _ORIGINAL_EIGH_P22
assert _returned_refit_p22 is _model_p22 and len(_controlled_calls_p22) == 1
assert np.array_equal(_X_refit_p22, _before_refit_p22)
_sentinel_order_p22 = np.argsort(_sentinel_values_p22)[::-1]
_expected_values_p22 = _sentinel_values_p22[_sentinel_order_p22]
_expected_components_p22 = _sentinel_vectors_p22[:, _sentinel_order_p22[:2]].T
assert np.array_equal(_model_p22.mean_, _mean_refit_p22)
assert np.array_equal(_model_p22.explained_variance_, _expected_values_p22[:2])
assert np.array_equal(
    _model_p22.explained_variance_ratio_,
    _expected_values_p22[:2] / _expected_values_p22.sum(),
)
assert np.allclose(
    _model_p22.components_.T @ _model_p22.components_,
    _expected_components_p22.T @ _expected_components_p22,
    atol=ATOL, rtol=RTOL,
)
_probe_refit_p22 = np.array([[44.0, -9.0, 15.0], [20.0, 2.0, -3.0]])
_probe_before_p22 = _probe_refit_p22.copy()
assert np.allclose(
    _model_p22.transform(_probe_refit_p22),
    (_probe_refit_p22 - _mean_refit_p22) @ _model_p22.components_.T,
    atol=ATOL, rtol=RTOL,
)
assert np.array_equal(_probe_refit_p22, _probe_before_p22)

_state_before_failed_refit_p22 = {
    name: getattr(_model_p22, name).copy()
    for name in ("mean_", "components_", "explained_variance_", "explained_variance_ratio_")
}
_scores_before_failed_refit_p22 = _model_p22.transform(_probe_refit_p22).copy()
_nonfinite_returns_p22 = (
    (np.array([np.nan, 2.0, 1.0]), np.eye(3)),
    (np.array([3.0, 2.0, 1.0]), np.array([[1.0, 0.0, 0.0], [0.0, np.inf, 0.0], [0.0, 0.0, 1.0]])),
)
for _bad_values_p22, _bad_vectors_p22 in _nonfinite_returns_p22:
    _nonfinite_calls_p22 = []
    def _nonfinite_eigh_p22(matrix):
        _nonfinite_calls_p22.append(np.array(matrix, copy=True))
        assert np.allclose(matrix, _C_ref_p22, atol=ATOL, rtol=RTOL)
        return _bad_values_p22.copy(), _bad_vectors_p22.copy()
    np.linalg.eigh = _nonfinite_eigh_p22
    try:
        _model_p22.fit(_X_train_p22)
    except ValueError:
        pass
    else:
        raise AssertionError("non-finite eigenpairs must raise ValueError")
    finally:
        np.linalg.eigh = _ORIGINAL_EIGH_P22
    assert len(_nonfinite_calls_p22) == 1
    for _name_atomic_p22, _old_atomic_p22 in _state_before_failed_refit_p22.items():
        assert np.array_equal(getattr(_model_p22, _name_atomic_p22), _old_atomic_p22)
    assert np.array_equal(_model_p22.transform(_probe_refit_p22), _scores_before_failed_refit_p22)

_invalid_fit_p22 = (
    (np.ones(3), 1),
    (np.ones((1, 3)), 1),
    (np.ones((3, 0)), 1),
    (np.array([[1.0, np.nan], [2.0, 3.0]]), 1),
    (np.array([[1e200, -1e200, 1e200], [-1e200, 1e200, -1e200], [0.0, 0.0, 0.0]]), 1),
    (np.ones((3, 2)), 0),
    (np.ones((3, 2)), 3),
    (np.ones((3, 2)), True),
    (np.array([["a", "b"], ["c", "d"]]), 1),
)
for _X_bad_p22, _k_bad_p22 in _invalid_fit_p22:
    _bad_calls_p22 = []
    def _unexpected_eigh_p22(matrix):
        _bad_calls_p22.append(np.array(matrix, copy=True))
        return _ORIGINAL_EIGH_P22(matrix)
    np.linalg.eigh = _unexpected_eigh_p22
    try:
        with np.errstate(over="ignore", invalid="ignore"):
            NumpyPCA(_k_bad_p22).fit(_X_bad_p22)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid fit input must raise ValueError")
    finally:
        np.linalg.eigh = _ORIGINAL_EIGH_P22
    assert _bad_calls_p22 == []

for _X_bad_transform_p22 in (np.ones(3), np.empty((0, 3)), np.ones((2, 4)), np.array([[1.0, np.inf, 2.0]])):
    try:
        _model_p22.transform(_X_bad_transform_p22)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid transform input must raise ValueError")
try:
    NumpyPCA(1).transform(np.ones((2, 3)))
except ValueError:
    pass
else:
    raise AssertionError("transform before fit must raise ValueError")

_constant_p22 = np.full((4, 3), 7.0)
_zero_model_p22 = NumpyPCA(2)
_fit_with_eigh_trace_p22(_zero_model_p22, _constant_p22)
assert np.allclose(_zero_model_p22.explained_variance_, 0.0, atol=ATOL, rtol=RTOL)
assert np.array_equal(_zero_model_p22.explained_variance_ratio_, np.zeros(2))